# BAF variants on Google Colab (A100): robustness analysis

This notebook runs the robustness analysis on the five variants of the Bank Account Fraud suite
(Variant I to V), with the same protocol as the main study (temporal split, undersampling, seeds,
preprocessing and metrics) but only with 10,000 training rows and the full training pool, and with
a subset of models:

* **Part A** (Python 3.9 environment): TabPFN v2 (10,000 rows only), XGBoost (default and tuned),
  random forest and L1 logistic regression (default).
* **Part B** (Colab's Python): TabPFN-3.5.

**Before starting**
1. Download `Variant I.csv` ... `Variant V.csv` from the BAF Kaggle page into `data/`.
2. On your computer, run `python exportar_para_colab.py --conjunto v1` (and `v2` ... `v5`). This
   creates `paquete_colab/variantes/v1` ... `v5` with the data, partitions and fingerprints, and copies
   the scripts to `paquete_colab/`.
3. Copy `paquete_colab/` to `My Drive/paquete_colab` in Google Drive (if the folder already exists
   from the main study, it is enough to add the `variantes/` folder and replace the three scripts).
4. In Colab: *Runtime → Change runtime type → A100 GPU* (and high RAM).
5. For Part B, the Colab secret `TABPFN_TOKEN` must exist (as in the main study).

Every script saves one row per fitted model in Drive and resumes where it stopped if the session
is interrupted (re-run cell 1 and the installation cell of the part you were running, then the
cell that was interrupted).

## 1. Mount Google Drive and check the machine

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc; grep -m1 'model name' /proc/cpuinfo
%cd /content/drive/MyDrive/paquete_colab
!ls variantes

# Part A: TabPFN v2 and classical models
## 2. Python 3.9 environment with the versions of the main study (about 3 minutes)

In [ ]:
!pip -q install uv
!uv venv -q /content/venv39 --python 3.9
!uv pip install -q --python /content/venv39/bin/python --extra-index-url https://download.pytorch.org/whl/cu118 --index-strategy unsafe-best-match numpy==1.24.4 pandas==2.3.1 scipy==1.13.1 scikit-learn==1.6.1 joblib==1.5.1 threadpoolctl==3.6.0 xgboost==2.1.4 tensorflow==2.11.0 torch==2.7.1+cu118 tabpfn==2.1.3 huggingface-hub==0.36.1 einops==0.8.1
!/content/venv39/bin/python -c "import sys, importlib.metadata as m; print(sys.version.split()[0], {p: m.version(p) for p in ['numpy','pandas','scikit-learn','xgboost','tensorflow','torch','tabpfn']})"

## 3. Quick check on Variant I (repetition 0, 10,000 rows, a few minutes)
Its rows go to `variantes/v1/resultados_tiempos_colab_prueba.csv` and are not used.

In [ ]:
%cd /content/drive/MyDrive/paquete_colab
!/content/venv39/bin/python tiempos_colab.py --grupo variantes --carpeta variantes/v1 --prueba-rapida

## 4. Full run of Part A on the five variants (10 repetitions × 2 sizes; about 20–30 minutes per variant)
Output: `variantes/<v>/resultados_tiempos_colab.csv`.

In [ ]:
%cd /content/drive/MyDrive/paquete_colab
for v in ['v1', 'v2', 'v3', 'v4', 'v5']:
    print('=' * 30, v, '=' * 30)
    !/content/venv39/bin/python tiempos_colab.py --grupo variantes --carpeta variantes/{v}

# Part B: TabPFN-3.5
## 5. Install TabPFN 9.1.0 in Colab's Python and load the token

In [ ]:
!pip -q install tabpfn==9.1.0
import os
from google.colab import userdata
os.environ['TABPFN_TOKEN'] = userdata.get('TABPFN_TOKEN')
import importlib.metadata as m
print('tabpfn', m.version('tabpfn'), '| torch', m.version('torch'), '| token loaded:', bool(os.environ['TABPFN_TOKEN']))

## 6. TabPFN-3.5 on the five variants (10 repetitions × 2 sizes; about 70–80 minutes per variant)
`--solo-verificar` first checks the data and every preprocessed test matrix against the fingerprints.
Output: `variantes/<v>/resultados_tabpfn35.csv`.

In [ ]:
%cd /content/drive/MyDrive/paquete_colab
for v in ['v1', 'v2', 'v3', 'v4', 'v5']:
    print('=' * 30, v, '=' * 30)
    !python tabpfn35_colab.py --carpeta variantes/{v} --tamanos 10000,completo --solo-verificar
    !python tabpfn35_colab.py --carpeta variantes/{v} --tamanos 10000,completo

## 7. When finished
Download the folder `My Drive/paquete_colab/variantes` (without the large `datos_baf.npz` files and
`puntajes/` folders) and place it in the repository as `resultados_colab/variantes/`. For each variant it
contains `resultados_tiempos_colab.csv`, `resultados_tabpfn35.csv`, `huellas.json`, the logs and the
environment files (`entorno_tiempos_variantes.json`, `entorno_colab.json`).

# Ablation: Variant V without x1 and x2
## 8. Same rows as Variant V, without x1 and x2 (10,000 rows only; about 15 minutes)
Requires the folder `variantes/v5sx` (from `python exportar_para_colab.py --conjunto v5sx`) and the
current `tiempos_colab.py`. Run cell 1 first. Part A needs the Python 3.9 environment (cell 2) and
Part B needs TabPFN 9.1.0 and the token (cell 5); run them in this order.

In [ ]:
# 8a. Part A (after cell 2)
%cd /content/drive/MyDrive/paquete_colab
!/content/venv39/bin/python tiempos_colab.py --grupo variantes --carpeta variantes/v5sx

In [ ]:
# 8b. Part B (after cell 5)
%cd /content/drive/MyDrive/paquete_colab
!python tabpfn35_colab.py --carpeta variantes/v5sx --tamanos 10000 --solo-verificar
!python tabpfn35_colab.py --carpeta variantes/v5sx --tamanos 10000